# Measure the grader

Python 3.12 · offline after setup · estimated time: 18 minutes

You will join the two prior artifacts, classify every case, calculate precision and recall, and persist an auditable `grader_report.json`.

Replace every `TODO`, then run each cell in order. Each visible check confirms one capability before you continue.


In [ ]:
# Setup: Install every pinned dependency into this notebook's Python environment.
import subprocess
import sys
from pathlib import Path

root = Path.cwd()
while root != root.parent and not (root / 'build/lesson-03/requirements.txt').is_file():
    root = root.parent
requirements_path = root / 'build/lesson-03/requirements.txt'
if not requirements_path.is_file():
    raise FileNotFoundError('Run this notebook from inside the course directory')
pip_command = [sys.executable, '-m', 'pip', 'install']
if sys.prefix == sys.base_prefix:
    pip_command.append('--user')
subprocess.check_call([*pip_command, '-r', str(requirements_path)])


## Step 1 — Load the evidence

Load the human-labeled cases and judge predictions. The positive class remains `1 = regression present`; the grader must not change that meaning.


In [1]:
import json
from pathlib import Path

ROOT = Path.cwd()
while not (ROOT / 'build/lesson-01/eval_cases.jsonl').exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent

cases = [json.loads(line) for line in (ROOT / 'build/lesson-01/eval_cases.jsonl').read_text(encoding='utf-8').splitlines()]
predictions = [json.loads(line) for line in (ROOT / 'build/lesson-02/judge_predictions.jsonl').read_text(encoding='utf-8').splitlines()]
# Check 1: Confirm both earlier lessons produced enough records for grading.
assert len(cases) >= 10 and len(predictions) >= 10, 'Lesson 03 requires at least 10 records in each artifact'
print(f'CHECK 1 — loaded {len(cases)} cases and {len(predictions)} predictions')
print('CHECK 1 — positive class: label 1 = regression present')


CHECK 1 — loaded 10 cases and 10 predictions
CHECK 1 — positive class: label 1 = regression present


## Step 2 — Join by identity

Complete `TODO 1`. Your index must reject an empty or duplicate `case_id` and a non-binary human label. Then require exact ID coverage and unchanged copied labels before building `joined`.


In [2]:
# TODO 1: Index rows by case_id and reject invalid evidence.
def index(rows, name):
    result = {}
    # Add validation and store each row under its case_id.
    return result

# Check 2: Join both artifacts by unique case ID and verify labels and exact coverage.
case_by_id = index(cases, 'cases')
prediction_by_id = index(predictions, 'predictions')
assert case_by_id and prediction_by_id, 'TODO 1: index both artifacts by case_id'
assert set(case_by_id) == set(prediction_by_id), 'case-ID coverage differs'

joined = []
for case_id, case in case_by_id.items():
    prediction = prediction_by_id[case_id]
    assert prediction.get('human_label') == case['human_label'], f'{case_id}: copied human label differs'
    assert prediction.get('judge_label') in (0, 1), f'{case_id}: judge label must be binary'
    joined.append((case_id, case['human_label'], prediction['judge_label']))
print(f'CHECK 2 — joined {len(joined)} records by case_id with exact coverage: PASS')


AssertionError: TODO 1: index both artifacts by case_id

## Step 3 — Classify and count TP, FP, FN, and TN

Complete `TODO 2`. First map all four `(human_label, judge_label)` pairs to TP, FP, FN, or TN. Then classify every joined case and write the code that counts each outcome. The conservation check proves that every case was counted exactly once.


In [ ]:
# TODO 2A: Map every (human_label, judge_label) pair to its outcome.
OUTCOME_BY_LABELS = {}
assert set(OUTCOME_BY_LABELS) == {(1, 1), (0, 1), (1, 0), (0, 0)}, 'TODO 2A: map all four label pairs'
assert set(OUTCOME_BY_LABELS.values()) == {'TP', 'FP', 'FN', 'TN'}, 'TODO 2A: use each outcome exactly once'

# TODO 2B: Classify every joined case, then count TP, FP, FN, and TN.
outcomes = {}
counts = {}

# Check 3: Ensure every case maps to exactly one confusion-matrix outcome.
assert len(outcomes) == len(joined), 'TODO 2B: classify every joined case'
assert set(counts) == {'TP', 'FP', 'FN', 'TN'}, 'TODO 2B: count all four outcomes'
assert sum(counts.values()) == len(joined), 'every case must be counted exactly once'

# Plot rows as human labels and columns as judge labels.
import matplotlib.pyplot as plt
matrix = [[counts['TN'], counts['FP']], [counts['FN'], counts['TP']]]
labels = [['TN', 'FP'], ['FN', 'TP']]
fig, ax = plt.subplots(figsize=(5, 4))
ax.imshow(matrix, cmap='Blues')
ax.set(xlabel='Judge label', ylabel='Human label', title='Confusion matrix', xticks=(0, 1), yticks=(0, 1))
for row in range(2):
    for column in range(2):
        ax.text(column, row, f'{labels[row][column]}\n{matrix[row][column]}', ha='center', va='center')
plt.show()
print('CHECK 3 — outcomes:', ', '.join(f'{case_id}={outcomes[case_id]}' for case_id, _, _ in joined))
print('CHECK 3 — TP, FP, FN, TN count conservation: PASS', counts)


## Step 4 — Turn the formulas into code

Complete `TODO 3` yourself. Translate the formulas directly: precision is `TP / (TP + FP)` and recall is `TP / (TP + FN)`. For each metric, check its denominator first and return `0.0` when it is zero.


In [ ]:
# TODO 3: Translate both formulas into Python.
# precision = TP / (TP + FP), or 0.0 when TP + FP is zero
# recall = TP / (TP + FN), or 0.0 when TP + FN is zero
precision_denominator = counts['TP'] + counts['FP']
recall_denominator = counts['TP'] + counts['FN']

precision = None  # Replace with the precision formula and zero-division branch.
recall = None     # Replace with the recall formula and zero-division branch.

# Check 4: Compare the calculated counts, precision, and recall with the expected result.
assert precision is not None and recall is not None, 'TODO 3: calculate precision and recall from the formulas'
expected = {'TP': 3, 'FP': 1, 'FN': 2, 'TN': 4, 'precision': 0.75, 'recall': 0.6}
assert counts == {key: expected[key] for key in counts}
assert precision == expected['precision'] and recall == expected['recall']
print(f'CHECK 4 — precision={precision} recall={recall}; zero-division policy implemented: PASS')


## Step 5 — Persist the checked report

Only checked values reach the artifact. Write the report, reload it, and compare it with the in-memory object before declaring the capstone complete.


In [ ]:
report = {'positive_class': 'human_label = 1 means regression present', 'zero_division_policy': '0.0', 'case_count': len(joined), **counts, 'precision': precision, 'recall': recall}
artifact_path = ROOT / 'build/lesson-03/grader_report.json'
artifact_path.parent.mkdir(parents=True, exist_ok=True)
# Check 5: Save the final report, reload it, and verify reproducible output.
artifact_path.write_text(json.dumps(report, indent=2) + '\n', encoding='utf-8')
assert json.loads(artifact_path.read_text(encoding='utf-8')) == report
print(f"CHECK 5 — TP={counts['TP']} FP={counts['FP']} FN={counts['FN']} TN={counts['TN']} precision={precision} recall={recall}")
print(f'CHECK 5 — wrote and reloaded: {artifact_path.relative_to(ROOT)}')
print('FINAL PASS — grader report is reproducible offline')
